**This notebook contains code to extract relevant data from the metabolites xml file and match to the corresponding H-NMR spectra.**

**1. Build a list of HMDB ID's that have 1D HNMR data, based on filenames.**

In [25]:
import os, re, json, csv

root = "data/hmdb_nmr_peak_lists"

id_re = re.compile(r'HMDB[\s_-]*(\d{5,7})', re.IGNORECASE)

def norm_id(d): return f"HMDB{int(d):07d}"

def looks_like_oned_h1(filepath):
    """
    Heuristics:
      1) filenames containing 'nmroned' -> accept
      2) otherwise, peek header/body:
         - reject if it mentions 'F1' AND 'F2' (2D)
         - accept if it contains '1H' and not '13C'/'15N' in axis/nucleus lines
         - quick numeric sniff: lines with two floats -> likely 2D; single float ppm -> likely 1D
    """
    fn = os.path.basename(filepath).lower()
    if "nmroned" in fn:
        return True
    if "nmrtwod" in fn:
        return False

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read(20000)
    except Exception:
        return False

    low = text.lower()
    if "f1" in low and "f2" in low:
        return False
    # nucleus hints
    if "13c" in low or "15n" in low:
        return False
    if "1h" in low or "proton" in low:
        pass  # keep checking

    # simple numeric sniff: count lines with 1 vs 2+ floats
    one_cols = two_plus_cols = 0
    for line in text.splitlines():
        toks = re.findall(r'[-+]?\d*\.\d+|\d+', line)
        if not toks: 
            continue
        # ignore very long lines (headers)
        if len(toks) == 1:
            one_cols += 1
        elif len(toks) >= 2:
            two_plus_cols += 1
        if one_cols + two_plus_cols > 50:
            break
    # If mostly single-column numbers -> likely 1D peak list
    return one_cols >= max(10, two_plus_cols * 2)

keep_ids = set()
rows = []
excluded = []
total_txt = 0

for dirpath, _, files in os.walk(root):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        total_txt += 1
        full = os.path.join(dirpath, fn)

        if not looks_like_oned_h1(full):
            excluded.append(os.path.relpath(full, root))
            continue

        # find HMDB id (filename first, then content)
        m = id_re.search(fn)
        digits = None
        if not m:
            try:
                with open(full, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read(20000)
                m = id_re.search(text)
            except Exception:
                m = None

        if m:
            digits = m.group(1)
            keep_ids.add(norm_id(digits))
            rows.append([os.path.relpath(full, root), norm_id(digits), "oned_h1"])
        else:
            excluded.append(os.path.relpath(full, root))

with open("keep_ids_oned_h1.json", "w") as f:
    json.dump(sorted(keep_ids), f, indent=2)

with open("oned_h1_file_map.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["file","hmdb_id","type"]); w.writerows(rows)

'''with open("excluded_non_oned_or_noid.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(excluded))'''

print("Total .txt files scanned:", total_txt)
print("Unique HMDB IDs with 1D 1H data:", len(keep_ids))
print("Example kept IDs:", list(sorted(keep_ids))[:10])



Total .txt files scanned: 1896
Unique HMDB IDs with 1D 1H data: 892
Example kept IDs: ['HMDB0000001', 'HMDB0000002', 'HMDB0000005', 'HMDB0000008', 'HMDB0000010', 'HMDB0000011', 'HMDB0000012', 'HMDB0000014', 'HMDB0000016', 'HMDB0000017']


**2. Match existing 1D H-NMR spectra files to relevant information in the metabolite xml file.**

Creates a file called hmdb_subset_classes.csv

In [26]:
from lxml import etree as ET
import csv, json

xml_path = "data/hmdb_metabolites/hmdb_metabolites.xml"          # or .xml.gz if gzipped (use gzip.open)
ids_path = "keep_ids_oned_h1.json"
out_csv  = "hmdb_subset_classes.csv"

with open(ids_path) as f:
    keep_ids = set(json.load(f))

NS = "{http://www.hmdb.ca}"  # HMDB default namespace

def txt(parent, tag):
    """Safe text getter for direct child"""
    if parent is None: 
        return ""
    el = parent.find(NS + tag)
    return el.text.strip() if el is not None and el.text else ""

# Stream on end-of-element for <metabolite> to keep memory low
context = ET.iterparse(xml_path, events=("end",), tag=NS + "metabolite")

with open(out_csv, "w", newline="", encoding="utf-8") as fout:
    w = csv.writer(fout)
    w.writerow(["accession","name","kingdom","super_class","class","sub_class"])

    for _, metab in context:
        accession = txt(metab, "accession")
        if accession in keep_ids:
            name = txt(metab, "name")
            taxonomy = metab.find(NS + "taxonomy")
            row = [
                accession,
                name,
                txt(taxonomy, "kingdom"),
                txt(taxonomy, "super_class"),
                txt(taxonomy, "class"),
                txt(taxonomy, "sub_class"),   # <- the level you’ll use most
            ]
            w.writerow(row)

        # --- free memory ---
        metab.clear()
        # remove processed siblings from the tree to keep memory bounded
        while metab.getprevious() is not None:
            del metab.getparent()[0]

print(f"Done. Wrote classifications for {out_csv}")


Done. Wrote classifications for hmdb_subset_classes.csv


**Explore super_class**

Creates a file called super_class_counts.csv which contains a list of the super_classes and counts for each.

In [27]:
import pandas as pd

# Load the file
df = pd.read_csv("hmdb_subset_classes.csv")

# Count unique super_classes
unique_count = df["super_class"].nunique()
print(f"Number of unique super_classes: {unique_count}")

# Count compounds per super_class
counts = df["super_class"].value_counts()
print("\nCompounds per super_class:")
print(counts.to_string())

# Optionally, save to file
counts.to_csv("super_class_counts.csv")


Number of unique super_classes: 11

Compounds per super_class:
super_class
Organic acids and derivatives              231
Lipids and lipid-like molecules            192
Organoheterocyclic compounds               134
Organic oxygen compounds                   105
Benzenoids                                  96
Nucleosides, nucleotides, and analogues     55
Phenylpropanoids and polyketides            39
Organic nitrogen compounds                  29
Alkaloids and derivatives                    4
Organosulfur compounds                       4
Hydrocarbons                                 1


**Create "group"**

Merges some super_classes and creates a new column called group. Generates a file called hmdb_subset_super_groups.csv and a file called group_counts.csv.

In [28]:
import pandas as pd

# --- settings ---
in_csv   = "hmdb_subset_classes.csv"
out_csv  = "hmdb_subset_super_groups.csv"
counts_csv = "group_counts.csv"
RARE_THRESHOLD = 10   # super_classes with < 10 entries → "Other"

# 1) Load
df = pd.read_csv(in_csv)

# 2) Build group from super_class
df["group"] = df["super_class"].fillna("Other")

# 3) Collapse rare groups into "Other"
super_counts = df["group"].value_counts()
rare_groups = set(super_counts[super_counts < RARE_THRESHOLD].index)
df.loc[df["group"].isin(rare_groups), "group"] = "Other"

# 4) Save outputs
df.to_csv(out_csv, index=False)
df["group"].value_counts().to_csv(counts_csv)

# 5) Print summary
total = len(df)
group_counts = df["group"].value_counts()
print("Groups and counts:\n", group_counts.to_string())
print(f"\nTotal compounds: {total}")
print(f"Unique groups: {df['group'].nunique()}")
print(f"Rare groups merged into 'Other' (threshold < {RARE_THRESHOLD}): {sorted(rare_groups)}")
print(f"\nWrote:\n - {out_csv}\n - {counts_csv}")


Groups and counts:
 group
Organic acids and derivatives              231
Lipids and lipid-like molecules            192
Organoheterocyclic compounds               134
Organic oxygen compounds                   105
Benzenoids                                  96
Nucleosides, nucleotides, and analogues     55
Phenylpropanoids and polyketides            39
Organic nitrogen compounds                  29
Other                                        9

Total compounds: 890
Unique groups: 9
Rare groups merged into 'Other' (threshold < 10): ['Alkaloids and derivatives', 'Hydrocarbons', 'Organosulfur compounds']

Wrote:
 - hmdb_subset_super_groups.csv
 - group_counts.csv
